In [1]:
#%pip install azure-ai-ml

In [2]:
from azure.ai.ml import MLClient, command, dsl, Input, Output
from azure.ai.ml.entities import RecurrenceTrigger, RecurrencePattern, JobSchedule, Environment
from azure.identity import DefaultAzureCredential

# 1. Authenticate
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id="26b57ef9-1628-4837-a014-81f6424512c1",
    resource_group_name="rg-trading-bot-dev",
    workspace_name="ml-trading-workspace"
)

# 2. Define Environment
ml_env = Environment(
    name="trading-cluster-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml"
)

# 3A. Define STEP 1: The ETL Component
etl_component = command(
    code="./",  
    # THE FIX: If python succeeds, we use 'echo' to create a dummy file to signal Step 2.
    command="python etl_pipeline.py && echo 'done' > ${{outputs.signal_file}}",
    environment=ml_env,
    compute="trading-dev-machine",
    display_name="1_Data_Engineering_ETL",
    outputs={"signal_file": Output(type="uri_file")} # Declare the output
)

# 3B. Define STEP 2: The AI Component
ai_component = command(
    code="./",  
    command="python ai_agent.py", # Notice the Python script ignores the dummy input!
    environment=ml_env,
    compute="trading-dev-machine",
    display_name="2_Agentic_Thesis_Gen",
    inputs={"wait_for_signal": Input(type="uri_file")}, # Declare the dependency
    environment_variables={
        "KEY_VAULT_URL": "https://kv-ml-trading-workspace.vault.azure.net/"
    }
)

# 4. Construct the Dependency Graph
@dsl.pipeline(
    name="decoupled_quant_pipeline",
    description="2-Step Architecture: ETL followed by AI Generation"
)
def my_quant_pipeline():
    # Instantiate Step 1
    step_1 = etl_component()
    
    # ENFORCE THE DEPENDENCY: Pass Step 1's dummy output directly into Step 2
    step_2 = ai_component(wait_for_signal=step_1.outputs.signal_file)

pipeline_job = my_quant_pipeline()

# 5. Schedule It
job_schedule = JobSchedule(
    name="daily_market_clustering_schedule",
    trigger=RecurrenceTrigger(
        frequency="day",
        interval=1,
        schedule=RecurrencePattern(hours=21, minutes=30)
    ),
    create_job=pipeline_job
)

ml_client.schedules.begin_create_or_update(job_schedule)
print("✅ Multi-Step MLOps Schedule created with Dummy Signal dependency!")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

✅ Multi-Step MLOps Schedule created with Dummy Signal dependency!
..